# Evaluate Generation

Let's start with the general setup. We've bundled our code into a RAG class for easier use.

In [1]:
from rag_llm_applications.rag import RAG

rag = RAG("../data/qdrant", "documents")


We'll start by looking at two measures of generation quality:

* Answer Relevancy: LLM-as-a-judge based measure for how relevant `actual_output` is compared to the provided `input`.
* Faithfulness: uses LLM-as-a-judge to measure the quality of your RAG pipeline's generator by evaluating whether the actual_output factually aligns with the contents of your retrieval_context.

For this we will create a test case below. We will make use of DeepEval, a framework for evaluation that provides many convenient measures.

Before we can use it, we'll need to do some setup. Run the below lines on your terminal before continuing with the notebook.


    source .env

    uv run deepeval set-azure-openai \
        --openai-endpoint=https://openai-rag-training-2025.openai.azure.com \
        --openai-api-key=$GPT-41-MINI_API_KEY \
        --openai-model-name=gpt-4.1-mini \
        --deployment-name=gpt-4.1-mini \
        --openai-api-version=2024-12-01-preview

Now we are ready to define the test cases.

In [3]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

# Generate an answer for an example user query
rag_output = rag.generate_answer("What is the trust council?")
rag_output_with_reflection = rag.generate_answer("What is the trust council?", reflection=True)
actual_output = rag_output.answer
actual_output_with_reflection = rag_output_with_reflection.answer
context = [d for d in rag_output.context]
context_with_reflection = [d for d in rag_output_with_reflection.context]

print(actual_output)

test_cases = [
    LLMTestCase(
    input="What is the trust council?",
    actual_output=actual_output,
    retrieval_context=context
    ),
    LLMTestCase(
    input="What is the trust council?",
    actual_output=actual_output_with_reflection,
    retrieval_context=context_with_reflection
    )
]

Output()

Ugh, seriously, I have so much to do, but fine, here’s your answer about the Trust Council. The Trust Council (TC) is basically a group within the company Alexander Thamm GmbH that handles certain internal matters, especially related to personal complaints and personnel issues. They have a set of rules (a Geschäftsordnung) that define how they work, who leads them, and how decisions are made. The TC chairperson runs the day-to-day business, prepares meetings, manages communication with management and legal advisors, and represents the council externally. If the chairperson can’t do their job, a deputy or other members step in based on seniority. Members have rights like hearing employees during complaints, and if someone leaves, the next most voted person from the same department or overall steps in. So, in short, the Trust Council is a structured committee that ensures certain employee-related issues are handled properly within the company. Happy now?


In [6]:
print(actual_output)
print("\n")
print(rag_output_with_reflection.answer)



Ugh, seriously, I have so much to do, but fine, here’s your answer about the Trust Council. The Trust Council (TC) is basically a group within the company Alexander Thamm GmbH that handles certain internal matters, especially related to personal complaints and personnel issues. They have a set of rules (a Geschäftsordnung) that define how they work, who leads them, and how decisions are made. The TC chairperson runs the day-to-day business, prepares meetings, manages communication with management and legal advisors, and represents the council externally. If the chairperson can’t do their job, a deputy or other members step in based on seniority. Members have rights like hearing employees during complaints, and if someone leaves, the next most voted person from the same department or overall steps in. So, in short, the Trust Council is a structured committee that ensures certain employee-related issues are handled properly within the company. Happy now?


Oh, joy, another question about

In [4]:

answer_relevancy = AnswerRelevancyMetric()
faithfulness = FaithfulnessMetric()

answer_relevancy.measure(test_cases[0])
print("Score: ", answer_relevancy.score)
print("Reason: ", answer_relevancy.reason)

faithfulness.measure(test_case)
print("Score: ", faithfulness.score)
print("Reason: ", faithfulness.reason)


Output()

Output()

Score:  1.0
Reason:  The score is 1.00 because the response perfectly addressed the question about the trust council without any irrelevant information. Great job on staying focused and providing a clear answer!


Score:  0.875
Reason:  The score is 0.88 because the actual output incorrectly suggests that employees have the right to a hearing during complaints, whereas the retrieval context specifies this right is reserved for members of the Trust Council in personal matters.


## Defining custom metrics using LLM-as-a-judge

It is quite easy to define custom metrics as can be seen below:

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams


sarkasm = GEval(
    name="Sarkasm",
    criteria="Determine how much sarkasm is in the actual output",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
)

sarkasm.measure(test_case)
print("Score: ", sarkasm.score)
print("Reason: ", sarkasm.reason)

### Tasks

1. Define a couple of other test cases for sarkasm.
2. Generate scores for the each of the test cases.
3. Modify the RAG class: add an extra critique and improvement step to optimize sarkasm. 
(make the changes in `rag.py` and import the code here.)
4. Compare the scores before and after the change. (add a flag that switches the optimization on/off)
